# CRIMEX v1 — Reproducible LA Crime Enrichment Pipeline

This notebook demonstrates how to build a small, reproducible CRIMEX dataset from real Los Angeles crime open data.

The goal is to show how raw crime incident records can be transformed into an enriched crime intelligence dataset using the existing CRIMEX Python pipeline modules.

This notebook is designed for GitHub publication and beginner-friendly understanding.

Main feature groups created by the pipeline:

- Source provenance
- Incident core fields
- Temporal features
- Geo-coordinate validation
- Crime ontology normalization
- Behavioral intelligence features
- Contextual risk features
- Quality validation features
- Graph-inspired features
- Explainability fields

For GitHub reproducibility, this notebook uses a small sample.

To run the full dataset later, only the download limit needs to be changed.

In [1]:
# [1.1] Setup imports and project folders

import sys
from pathlib import Path

import pandas as pd

# Project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parent

# Add project root (NOT src) to path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FINAL_DIR = DATA_DIR / "final"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Final folder:", FINAL_DIR)

Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX
Raw folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\raw
Final folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final


## 2. Download a Small LA Crime Sample

We download a small sample from the Los Angeles open crime dataset.

This keeps the notebook fast and reproducible for GitHub users.

To run the full dataset later, increase or remove the `$limit` value.

In [2]:
# [2.1] Download LA Crime Open Data sample

# Small sample for reproducible GitHub notebook
SAMPLE_SIZE = 10_000

la_crime_url = (
    "https://data.lacity.org/resource/2nrs-mtv8.csv"
    f"?$limit={SAMPLE_SIZE}"
)

df_raw = pd.read_csv(la_crime_url)

raw_sample_path = RAW_DIR / "la_crime_sample_10000.csv"

df_raw.to_csv(
    raw_sample_path,
    index=False
)

print("Downloaded shape:", df_raw.shape)
print("Saved raw sample:", raw_sample_path)

df_raw.head()

Downloaded shape: (10000, 28)
Saved raw sample: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\raw\la_crime_sample_10000.csv


,dr_no,date_rptd,date_occ,time_occ,area,area_name,rpt_dist_no,part_1_2,crm_cd,crm_cd_desc,...,status,status_desc,crm_cd_1,crm_cd_2,crm_cd_3,crm_cd_4,location,cross_street,lat,lon
0,211507896,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


## 3. Inspect Raw Data

Before applying any pipeline, it is important to understand the raw dataset.

This step helps beginners see:

- Number of records and columns  
- Column names  
- Missing values  
- Data quality issues  

Understanding raw data is a key step in any data science or feature engineering pipeline.

In [3]:
# [3.1] Basic inspection

print("Shape:", df_raw.shape)

print("\nColumns:")
print(df_raw.columns.tolist())

Shape: (10000, 28)

Columns:
['dr_no', 'date_rptd', 'date_occ', 'time_occ', 'area', 'area_name', 'rpt_dist_no', 'part_1_2', 'crm_cd', 'crm_cd_desc', 'mocodes', 'vict_age', 'vict_sex', 'vict_descent', 'premis_cd', 'premis_desc', 'weapon_used_cd', 'weapon_desc', 'status', 'status_desc', 'crm_cd_1', 'crm_cd_2', 'crm_cd_3', 'crm_cd_4', 'location', 'cross_street', 'lat', 'lon']


In [4]:
# [3.2] Missing values overview

missing_counts = df_raw.isnull().sum()

missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing_counts)

Columns with missing values:
crm_cd_4          9999
crm_cd_3          9967
crm_cd_2          9175
cross_street      8427
weapon_used_cd    6513
weapon_desc       6513
mocodes           1413
vict_sex          1322
vict_descent      1322
premis_desc          2
dtype: int64


In [5]:
# [3.3] Preview raw records

df_raw.head()

,dr_no,date_rptd,date_occ,time_occ,area,area_name,rpt_dist_no,part_1_2,crm_cd,crm_cd_desc,...,status,status_desc,crm_cd_1,crm_cd_2,crm_cd_3,crm_cd_4,location,cross_street,lat,lon
0,211507896,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


## 4. Map Raw Data to CRIMEX Input Schema

The CRIMEX pipeline expects a standardized input format.

However, real-world datasets often have different column names and structures.

In this step, we map the LA Crime dataset columns to the CRIMEX schema.

### Why this step is important:

- Different datasets use different naming conventions  
- A unified schema allows us to reuse the same pipeline  
- This is a common real-world data engineering task  

### Example:

| Raw Column       | CRIMEX Column         |
|-----------------|----------------------|
| dr_no           | case_id              |
| crm_cd_desc     | crime_description    |
| date_occ        | date_occured         |
| lat/lon         | latitude/longitude   |

In [6]:
# [4.1] Map LA Crime columns to CRIMEX schema

column_mapping = {
    "dr_no": "case_id",
    "crm_cd_desc": "crime_description",
    "date_rptd": "date_reported",
    "date_occ": "date_occured",
    "time_occ": "time_occured",
    "lat": "latitude",
    "lon": "longitude",
    "mocodes": "modus_operandi",
    "weapon_desc": "weapon_description",
    "premis_desc": "premise_description",
    "vict_age": "victim_age",
    "vict_sex": "victim_sex",
    "vict_descent": "victim_ethnicity",
}

df_input = df_raw.rename(columns=column_mapping)

print("Mapped columns:")
print(df_input.columns.tolist())

Mapped columns:
['case_id', 'date_reported', 'date_occured', 'time_occured', 'area', 'area_name', 'rpt_dist_no', 'part_1_2', 'crm_cd', 'crime_description', 'modus_operandi', 'victim_age', 'victim_sex', 'victim_ethnicity', 'premis_cd', 'premise_description', 'weapon_used_cd', 'weapon_description', 'status', 'status_desc', 'crm_cd_1', 'crm_cd_2', 'crm_cd_3', 'crm_cd_4', 'location', 'cross_street', 'latitude', 'longitude']


In [7]:
# [4.2] Select only relevant CRIMEX input fields

selected_columns = [
    "case_id",
    "crime_description",
    "date_reported",
    "date_occured",
    "time_occured",
    "latitude",
    "longitude",
    "modus_operandi",
    "weapon_description",
    "premise_description",
    "victim_age",
    "victim_sex",
    "victim_ethnicity",
]

df_input = df_input[selected_columns]

print("Input shape:", df_input.shape)
df_input.head()

Input shape: (10000, 13)


,case_id,crime_description,date_reported,date_occured,time_occured,latitude,longitude,modus_operandi,weapon_description,premise_description,victim_age,victim_sex,victim_ethnicity
0,211507896,THEFT OF IDENTITY,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,34.2124,-118.4092,0377,NaN,SINGLE FAMILY DWELLING,31,M,H
1,201516622,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,34.1993,-118.4203,0416 0334 2004 1822 1414 0305 0319 0400,KNIFE WITH BLADE 6INCHES OR LESS,SIDEWALK,32,M,H
2,240913563,THEFT OF IDENTITY,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,34.1847,-118.4509,0377,NaN,SINGLE FAMILY DWELLING,30,M,W
3,210704711,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,34.0339,-118.3747,0344,NaN,STREET,47,F,A
4,201418201,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,33.9813,-118.4350,1300 0344 1606 2032,NaN,ALLEY,63,M,H


## 5. Apply the CRIMEX Enrichment Pipeline

In this step, we apply the CRIMEX pipeline to transform the raw input data into an enriched crime intelligence dataset.

The pipeline is implemented in reusable Python modules located in the `src/` folder.

### Why this is important:

Instead of writing all logic inside the notebook, we:

- Use modular, reusable code  
- Keep the notebook clean and readable  
- Ensure reproducibility across projects  

### What the pipeline does:

The pipeline automatically generates multiple feature groups:

- Temporal features (time-based patterns)
- Geo validation (coordinate cleaning)
- Crime ontology mapping (standardized categories)
- Behavioral intelligence (modus operandi analysis)
- Contextual features (risk and environment signals)
- Data quality checks
- Graph-inspired intelligence features
- Explainability outputs

This is the core transformation from raw data → intelligence dataset.

### Note about Geo/POI enrichment

This notebook runs the full offline CRIMEX feature pipeline.

Online Geo/POI enrichment is disabled here because it requires external services, long runtime, and API rate-limit handling. In CRIMEX v2, this step can be enabled using a local OpenStreetMap / Overpass backend.

In [9]:
# [5.1] Run CRIMEX pipeline

from src.config import PIPELINE_CONFIG
from src.pipeline import run_crimex_pipeline

PIPELINE_CONFIG.update({
    # Offline deterministic modules
    "enable_translation_module": True,
    "enable_weather_enrichment": True,
    "enable_graph_features": True,
    "enable_quality_validation": True,

    # External services are disabled for GitHub reproducibility.
    # Enable later when using local OpenStreetMap / Overpass stack.
    "enable_geo_enrichment": False,
    "enable_poi_enrichment": False,
})

print("Running CRIMEX offline enrichment pipeline...")

df_enriched = run_crimex_pipeline(
    df_input,
    PIPELINE_CONFIG
)

print("Enriched shape:", df_enriched.shape)
print("Number of columns:", len(df_enriched.columns))

Running CRIMEX offline enrichment pipeline...


Explainability: 100%|████████████████████████████████████████████████████████████████| 10/10 [01:17<00:00,  7.80s/step]

CRIMEX pipeline completed.
Final shape: (10000, 141)
Enriched shape: (10000, 141)
Number of columns: 141


## 6. Understand the Generated Feature Groups

The CRIMEX pipeline expands the raw crime dataset from a small set of source columns into a richer machine-learning-ready dataset.

The generated features can be grouped into:

1. **Temporal features**  
   Time-based variables such as year, month, weekday, hour, weekend flag, and season.

2. **Geo validation features**  
   Checks whether coordinates are valid and creates cleaned latitude/longitude fields.

3. **Crime ontology features**  
   Maps raw crime descriptions into broader standardized crime families such as property, violent, fraud/identity, or child-related.

4. **Behavioral intelligence features**  
   Derives signals from crime type, weapon usage, target context, and modus operandi complexity.

5. **Contextual risk features**  
   Captures opportunity, guardianship, holiday, weather proxy, strain, and routine activity risk.

6. **Quality validation features**  
   Adds completeness, consistency, enrichment confidence, and review priority indicators.

7. **Graph-inspired intelligence features**  
   Creates graph node keys and connectivity-based risk proxies.

8. **Explainability features**  
   Generates human-readable and machine-readable explanations for elevated risk signals.

In [10]:
# [6.1] Inspect generated columns by keyword groups

feature_groups = {
    "temporal": [
        col for col in df_enriched.columns
        if col in [
            "year", "quarter", "month", "month_name",
            "week_of_year", "day", "day_of_year",
            "day_of_week_number", "day_of_week_name",
            "hour", "is_weekend", "local_season_code"
        ]
    ],
    "geo": [
        col for col in df_enriched.columns
        if "latitude" in col
        or "longitude" in col
        or "coordinate" in col
        or "hemisphere" in col
    ],
    "ontology": [
        col for col in df_enriched.columns
        if "ontology" in col
        or "crime_family" in col
        or "crime_severity" in col
    ],
    "behavior": [
        col for col in df_enriched.columns
        if "behavior" in col
        or "sophistication" in col
        or "escalation" in col
        or "threat" in col
        or "persistence" in col
    ],
    "context": [
        col for col in df_enriched.columns
        if "context" in col
        or "holiday" in col
        or "weather" in col
        or "routine" in col
        or "guardianship" in col
        or "strain" in col
    ],
    "quality": [
        col for col in df_enriched.columns
        if "completeness" in col
        or "consistency" in col
        or "confidence" in col
        or "review" in col
    ],
    "graph": [
        col for col in df_enriched.columns
        if "node_key" in col
        or "connectivity" in col
        or "centrality" in col
        or "network" in col
        or "cooffending" in col
    ],
    "explainability": [
        col for col in df_enriched.columns
        if "explanation" in col
    ]
}

for group_name, columns in feature_groups.items():
    print(f"\n{group_name.upper()} ({len(columns)} columns)")
    print(columns)


TEMPORAL (12 columns)
['year', 'quarter', 'month', 'month_name', 'week_of_year', 'day', 'day_of_year', 'day_of_week_number', 'day_of_week_name', 'hour', 'is_weekend', 'local_season_code']

GEO (8 columns)
['latitude', 'longitude', 'latitude_original', 'longitude_original', 'is_valid_coordinate', 'latitude_final', 'longitude_final', 'hemisphere_code']

ONTOLOGY (6 columns)
['crime_ontology_code', 'crime_ontology_desc', 'crime_family_code', 'crime_family_desc', 'crime_severity_code', 'crime_severity_desc']

BEHAVIOR (25 columns)
['multi_step_behavior_flag', 'behavior_risk_profile_code', 'behavior_risk_profile_desc', 'criminal_sophistication_score', 'criminal_sophistication_code', 'behavior_signature_risk_score', 'behavior_signature_risk_code', 'behavior_signature_text', 'behavior_signature_key', 'behavior_signature_frequency', 'behavior_signature_rarity_code', 'escalation_potential_score', 'escalation_potential_code', 'behavioral_diversity_score', 'behavioral_diversity_code', 'behaviora

### Understanding the Feature Groups (Beginner-Friendly)

The CRIMEX pipeline transforms raw crime records into a richer dataset by adding multiple layers of intelligence.

Below is a simplified explanation of what each group represents:

---

### 🕒 Temporal Features (12)

These describe **when the crime happened**.

Examples:
- Hour of the day
- Day of the week
- Weekend vs weekday
- Season (winter, summer)

👉 Why useful?
Crimes often follow time patterns (e.g., theft at night, assaults on weekends).

---

### 🌍 Geo Features (8)

These validate and standardize **location information**.

Examples:
- Cleaned latitude/longitude
- Coordinate validity flag
- Hemisphere indicator

👉 Why useful?
Reliable location data is essential for mapping and spatial analysis.

---

### 🧩 Crime Ontology (6)

These convert raw crime descriptions into **standard categories**.

Examples:
- Property crime
- Violent crime
- Fraud / identity crime

👉 Why useful?
Raw text is inconsistent — categories make modeling easier.

---

### 🧠 Behavioral Intelligence (25)

These estimate **how the crime was committed**.

Examples:
- Multi-step crime indicator
- Criminal sophistication score
- Behavioral threat score
- Escalation potential

👉 Why useful?
Two crimes may look similar but differ in complexity and risk level.

---

### ⚙️ Contextual Risk Features (19)

These describe the **environment around the crime**.

Examples:
- Holiday proximity
- Weather stress proxy
- Opportunity and guardianship signals
- Routine activity risk

👉 Why useful?
Crime is influenced by context (time, environment, social patterns).

---

### 📊 Quality Features (5)

These evaluate **data reliability**.

Examples:
- Completeness score
- Consistency flag
- Confidence level

👉 Why useful?
Real-world data is messy — quality matters for ML models.

---

### 🔗 Graph-Inspired Features (14)

These simulate **network relationships**.

Examples:
- Node identifiers
- Connectivity scores
- Exposure risk

👉 Why useful?
Crime patterns often behave like networks, even without explicit links.

---

### 🧾 Explainability Features (2)

These help explain **why a record has a certain risk level**.

Examples:
- Risk explanation text
- Explanation codes

👉 Why useful?
Important for interpretable AI and decision support systems.

---

### Summary

From only ~13 raw columns, the dataset expands to **141 features**, turning simple records into a structured intelligence dataset ready for:

- Machine learning
- Risk scoring
- Pattern discovery
- Explainable AI

## 7. Save the Enriched Dataset

We save the final CRIMEX dataset in multiple formats for different use cases:

- **Parquet (full dataset)** → efficient, recommended for ML
- **CSV sample (small subset)** → easy for quick exploration
- **Schema (JSON)** → documents structure and ensures reproducibility

This step makes the dataset ready for:
- GitHub publication
- Hugging Face datasets
- Reproducible research

In [11]:
# [7.1] Save CRIMEX enriched dataset

import json

DATASET_NAME = "crimex_v1_la_sample"

parquet_path = FINAL_DIR / f"{DATASET_NAME}.parquet"
csv_sample_path = FINAL_DIR / f"{DATASET_NAME}_sample_2000.csv"
schema_path = FINAL_DIR / f"{DATASET_NAME}_schema.json"

# Save full dataset (parquet)
df_enriched.to_parquet(
    parquet_path,
    index=False
)

# Save small sample for GitHub preview
df_enriched.sample(
    n=2000,
    random_state=42
).to_csv(
    csv_sample_path,
    index=False
)

# Save schema
schema = {
    "dataset_name": DATASET_NAME,
    "rows": int(df_enriched.shape[0]),
    "columns": int(df_enriched.shape[1]),
    "column_names": df_enriched.columns.tolist()
}

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print("Saved parquet:", parquet_path)
print("Saved sample CSV:", csv_sample_path)
print("Saved schema:", schema_path)

Saved parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\crimex_v1_la_sample.parquet
Saved sample CSV: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\crimex_v1_la_sample_sample_2000.csv
Saved schema: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\crimex_v1_la_sample_schema.json


## 8. Final Summary

In this notebook, we demonstrated how to build a CRIMEX dataset from raw crime data.

### Pipeline steps:

1. Download raw data from LA Open Data  
2. Inspect and understand the dataset  
3. Map raw fields to a unified schema  
4. Apply the CRIMEX enrichment pipeline  
5. Generate advanced feature groups  
6. Save the dataset in reproducible formats  

### Final result:

- Input: ~13 raw fields  
- Output: **141 enriched features**  

### Why this matters:

This transformation enables:

- Machine learning models  
- Risk prediction systems  
- Behavioral analysis  
- Explainable AI  

### Scaling to full dataset:

To run on the full LA dataset (~1M records), simply modify:

```python
SAMPLE_SIZE = None  # remove limit or increase value